In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("StudentPerformanceFactors.csv")
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

In [4]:
for col in df.select_dtypes(include='object'):
    print(col, df[col].unique())

Parental_Involvement ['Low' 'Medium' 'High']
Access_to_Resources ['High' 'Medium' 'Low']
Extracurricular_Activities ['No' 'Yes']
Motivation_Level ['Low' 'Medium' 'High']
Internet_Access ['Yes' 'No']
Family_Income ['Low' 'Medium' 'High']
Teacher_Quality ['Medium' 'High' 'Low' nan]
School_Type ['Public' 'Private']
Peer_Influence ['Positive' 'Negative' 'Neutral']
Learning_Disabilities ['No' 'Yes']
Parental_Education_Level ['High School' 'College' 'Postgraduate' nan]
Distance_from_Home ['Near' 'Moderate' 'Far' nan]
Gender ['Male' 'Female']


## Encoding and handling missing values

In [5]:
# 1. Missing values fill karo
cols = ['Teacher_Quality', 
        'Parental_Education_Level', 
        'Distance_from_Home']

for col in cols:
    df[col] = df[col].fillna(df[col].mode()[0])


# 2. Ordinal encoding
df['Parental_Involvement'] = df['Parental_Involvement'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})

df['Access_to_Resources'] = df['Access_to_Resources'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})

df['Motivation_Level'] = df['Motivation_Level'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})

df['Family_Income'] = df['Family_Income'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})

df['Teacher_Quality'] = df['Teacher_Quality'].map({
    'Low': 0, 'Medium': 1, 'High': 2
})

df['Parental_Education_Level'] = df['Parental_Education_Level'].map({
    'High School': 0,
    'College': 1,
    'Postgraduate': 2
})

df['Distance_from_Home'] = df['Distance_from_Home'].map({
    'Near': 0,
    'Moderate': 1,
    'Far': 2
})


# 3. Binary encoding
df['Extracurricular_Activities'] = df['Extracurricular_Activities'].map({
    'No': 0, 'Yes': 1
})

df['Internet_Access'] = df['Internet_Access'].map({
    'No': 0, 'Yes': 1
})

df['School_Type'] = df['School_Type'].map({
    'Public': 0, 'Private': 1
})

df['Learning_Disabilities'] = df['Learning_Disabilities'].map({
    'No': 0, 'Yes': 1
})

df['Gender'] = df['Gender'].map({
    'Male': 0, 'Female': 1
})


# 4. Peer Influence → One Hot Encoding
df = pd.get_dummies(
    df,
    columns=['Peer_Influence'],
    drop_first=True,
    dtype=int
)

In [6]:
df.select_dtypes(include='object').columns

Index([], dtype='object')

#### Define X and y

In [7]:
X = df.drop('Exam_Score', axis=1)
y = df['Exam_Score']

In [8]:
print(X.shape)
print(y.shape)

(6607, 20)
(6607,)


#### Perform Train_test_split

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [10]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5285, 20)
(1322, 20)
(5285,)
(1322,)


#### Applying LinearRegression

In [11]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 0.4456866960207039
MSE: 3.240731877921472
R2 Score: 0.7707311452347383


#### Applying RandomForest

In [15]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
y_pred_1 = model.predict(X_test)

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred_1))
print("MSE:", mean_squared_error(y_test, y_pred_1))
print("R2 Score:", r2_score(y_test, y_pred_1))

#### Applying Gradien

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(random_state=42)

model.fit(X_train, y_train)
y_pred_2 = model.predict(X_test)

In [18]:
print("MAE:", mean_absolute_error(y_test, y_pred_2))
print("MSE:", mean_squared_error(y_test, y_pred_2))
print("R2 Score:", r2_score(y_test, y_pred_2))

MAE: 0.794499297037522
MSE: 3.7729433644785093
R2 Score: 0.7330793052762565


# model = LinearRegression

## conclution

### In this project, we used student information to predict their exam scores. We tried Linear Regression, Random Forest, and Gradient Boosting.

### Among these, Linear Regression gave the best results with an R² score of 77.1%. So, we selected Linear Regression as the best model for this project.

### Overall, the model can predict student exam scores fairly well.